# Exercice n°7 : Machine Learning pour la Neuroimagerie - Prédire le TDAH

Bienvenue dans cet exercice pratique. Vous allez construire un pipeline complet d'apprentissage automatique pour prédire le diagnostic de **Trouble du Déficit de l'Attention avec ou sans Hyperactivité (TDAH)** à partir de données de connectivité fonctionnelle IRM.

Vous utiliserez le jeu de données **ADHD200**, disponible via nilearn, qui contient des données d'IRMf de repos pour des sujets avec et sans diagnostic de TDAH.

Cet exercice vous fera appliquer les mêmes étapes que celles vues en classe :
1. Exploration des données phénotypiques
2. Extraction de features de connectivité fonctionnelle
3. Entraînement et évaluation d'un classifieur SVM

**Consignes importantes pour l'évaluation automatique :**

1. **Conformité** : Ne modifiez pas les noms des variables de résultat (ex: `q1_n_sujets`, `defi_accuracy`) donnés dans les lignes commentées. Ces noms sont essentiels pour l'évaluation automatique.
2. **Exécution** : Assurez-vous que votre code s'exécute sans erreur de haut en bas.
3. **Paramètres** : Respectez exactement les paramètres spécifiés dans chaque question (atlas, random_state, etc.). L'évaluateur automatique compare vos résultats à des valeurs de référence calculées avec ces paramètres précis.
4. **Données** : Utilisez uniquement les variables `pheno`, `func` et `confounds` fournies dans la cellule de configuration (Section 0), qui sont déjà alignées entre elles.

Bon courage !

## Section 0 : Configuration et chargement des données

Exécutez ces cellules de configuration. Elles téléchargent les données et préparent les variables que vous utiliserez dans l'exercice. **Ne les modifiez pas.**

In [ ]:
# Configuration — ne pas modifier
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from nilearn import datasets, plotting
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Bibliothèques importées avec succès.")

In [ ]:
# Chargement et alignement du dataset ADHD200 — ne pas modifier
data_dir = './nilearn_data'

adhd_dataset = datasets.fetch_adhd(n_subjects=40, data_dir=data_dir)
pheno_brut = pd.DataFrame(adhd_dataset.phenotypic)

def get_id(path):
    # Extrait l'ID numérique du nom de fichier (ex: 0010001)
    base = os.path.basename(path)
    return int(base.split('_')[0])

func_ids = [get_id(f) for f in adhd_dataset.func]
pheno_ids = set(pheno_brut['Subject'].values)

# Alignement automatique entre imagerie et phénotype
matched_idx = [i for i, fid in enumerate(func_ids) if fid in pheno_ids]
matched_fids = [func_ids[i] for i in matched_idx]

pheno = pheno_brut.set_index('Subject').loc[matched_fids].reset_index()
func = [adhd_dataset.func[i] for i in matched_idx]
confounds = [adhd_dataset.confounds[i] for i in matched_idx]

print(f"Sujets avec données d'imagerie ET phénotypiques : {len(func)}")

---
## Partie 1 : Exploration des données phénotypiques (25 points)

Dans cette section, vous allez explorer les variables cliniques et démographiques du jeu de données ADHD200.

### Question 1 : Nombre de sujets

En utilisant la variable `pheno` fournie dans la Section 0, déterminez le nombre total de sujets disposant à la fois de données d'imagerie et de données phénotypiques.

Stockez ce nombre entier dans **`q1_n_sujets`**.

In [ ]:
# Réponse 1

# Votre code ici

# q1_n_sujets = ...

### Question 2 : Distribution des diagnostics

La colonne `adhd` du DataFrame `pheno` contient le diagnostic : `1` pour TDAH, `0` pour contrôle.

Comptez le nombre de sujets TDAH et le nombre de sujets contrôles.

Stockez les résultats dans **`q2_n_tdah`** (entier) et **`q2_n_controles`** (entier).

In [ ]:
# Réponse 2

# Votre code ici

# q2_n_tdah = ...
# q2_n_controles = ...

### Question 3 : Variable cible

Créez le vecteur cible `y` pour l'apprentissage automatique : un tableau NumPy d'entiers (dtype `int`) contenant `1` pour les sujets TDAH et `0` pour les contrôles, dans l'ordre de la variable `pheno`.

Stockez le résultat dans **`q3_y`**.

In [ ]:
# Réponse 3

# Votre code ici

# q3_y = ...

### Question 4 : Âge moyen par groupe

Calculez l'âge moyen (colonne `age`) séparément pour les sujets TDAH et pour les contrôles.

Stockez l'âge moyen des sujets **TDAH** dans **`q4_age_moyen_tdah`** (float) et l'âge moyen des **contrôles** dans **`q4_age_moyen_controles`** (float).

In [ ]:
# Réponse 4

# Votre code ici

# q4_age_moyen_tdah = ...
# q4_age_moyen_controles = ...

### Question 5 : Répartition des sexes

Calculez le nombre de sujets masculins (`'M'`) dans chaque groupe diagnostique (TDAH vs contrôle).

Stockez le nombre d'hommes parmi les sujets **TDAH** dans **`q5_n_hommes_tdah`** (entier) et parmi les **contrôles** dans **`q5_n_hommes_controles`** (entier).

In [ ]:
# Réponse 5

# Votre code ici

# q5_n_hommes_tdah = ...
# q5_n_hommes_controles = ...

---
## Partie 2 : Extraction de features de connectivité fonctionnelle (20 points)

Dans cette section, vous allez extraire les matrices de connectivité fonctionnelle pour chaque sujet en utilisant un atlas cérébral. Ces matrices constitueront les **features** (variables prédictives) de votre modèle ML.

### Question 6 : Extraction des séries temporelles

Chargez l'atlas **BASC multiscale** à résolution **12 ROIs** (`resolution=12`).
Initialisez un `NiftiLabelsMasker` avec les paramètres :
- `labels_img=atlas_img`
- `standardize='zscore_sample'`
- `detrend=True`

Utilisez ce masker pour extraire les séries temporelles du **premier sujet** (`func[0]`, `confounds[0]`). Stockez les séries temporelles dans **`q6_timeseries`**.

Stockez le **nombre de régions d'intérêt (ROIs)** dans **`q6_n_rois`**.

In [ ]:
# Réponse 6
basc = datasets.fetch_atlas_basc_multiscale_2015(data_dir=data_dir, resolution=12)
atlas_img = basc.maps

# Votre code ici

# q6_timeseries = ...
# q6_n_rois = ...

### Question 7 : Matrice de connectivité vectorisée

En utilisant `ConnectivityMeasure` (`kind='correlation'`, `vectorize=True`, `discard_diagonal=True`), calculez le vecteur de connectivité pour le premier sujet à partir de `q6_timeseries`.

Stockez le vecteur dans **`q7_corr_vecteur`** et le **nombre de features** dans **`q7_n_features`**.

In [ ]:
# Réponse 7

# Votre code ici

# q7_corr_vecteur = ...
# q7_n_features = ...

### Question 8 : Extraction pour tous les sujets

Répétez l'extraction pour **tous les sujets** de la liste `func`. Construisez la matrice de features `X` en empilant les vecteurs de connectivité.

Stockez la matrice résultante dans **`q8_X_features`**.

In [ ]:
# Réponse 8

# Votre code ici

# q8_X_features = ...

### Question 9 : Vérification de la forme de la matrice

Vérifiez que la matrice `q8_X_features` a la bonne forme. 
Stockez la **forme (shape)** de la matrice dans **`q9_X_shape`** (tuple).

In [ ]:
# Réponse 9

# Votre code ici

# q9_X_shape = ...

---
## Partie 3 : Pipeline d'apprentissage automatique (25 points)

### Question 10 : Séparation entraînement / test

Divisez les données en un ensemble d'entraînement et un ensemble de test en utilisant `train_test_split` (`test_size=0.2`, `shuffle=True`, `stratify=q3_y`, `random_state=0`). 

Stockez les tailles dans **`q10_n_train`** et **`q10_n_test`**.

In [ ]:
# Réponse 10

# Votre code ici

# q10_n_train = ...
# q10_n_test = ...

### Question 11 : Standardisation des données

Appliquez un `StandardScaler` aux données d'entraînement et de test.
Stockez la **moyenne globale** de l'ensemble d'entraînement standardisé dans **`q11_moyenne_train_scl`**.

In [ ]:
# Réponse 11

# Votre code ici

# q11_moyenne_train_scl = ...

### Question 12 : Validation croisée

Initialisez un classifieur SVM (`kernel='linear'`, `class_weight='balanced'`). Obtenez les prédictions par validation croisée à **3 plis** (`cv=3`) sur l'ensemble d'entraînement.

Stockez les prédictions dans **`q12_y_pred_cv`**.

In [ ]:
# Réponse 12

# Votre code ici

# q12_y_pred_cv = ...

### Question 13 : Accuracy de la validation croisée

Calculez l'accuracy globale sur l'ensemble d'entraînement.
Stockez-la dans **`q13_cv_accuracy`**.

In [ ]:
# Réponse 13

# Votre code ici

# q13_cv_accuracy = ...

### Question 14 : Matrice de confusion

Calculez la matrice de confusion sur l'ensemble d'entraînement.
Stockez-la dans **`q14_cm`**.

In [ ]:
# Réponse 14

# Votre code ici

# q14_cm = ...

---
## Défi : Évaluation finale sur données non vues (30 points)

Entraînez le modèle sur l'ensemble d'entraînement complet et évaluez-le sur l'ensemble de test.

Stockez les prédictions dans **`defi_y_pred`** et l'accuracy finale dans **`defi_accuracy`**.

In [ ]:
# Défi

# Votre code ici

# defi_y_pred = ...
# defi_accuracy = ...

---
## Pour aller plus loin (non noté)

Si vous avez terminé, explorez ces pistes :

1. **Autre atlas** : refaites l'extraction avec `resolution=36` (630 features). Le modèle est-il meilleur ou pire ? Pourquoi ?
2. **Test de permutation** : utilisez `permutation_test_score` pour vérifier si la performance du modèle est statistiquement significative malgré le petit nombre de sujets.
3. **Visualisation des poids** : entraînez le modèle final, récupérez `svc.coef_` et utilisez `correlation_measure.inverse_transform` pour afficher les connexions les plus importantes sous forme de connectome.
4. **Régression** : remplacez le diagnostic binaire par une mesure continue (ex: score symptomatique `dsm_iv_tot`) et utilisez un `SVR` (Support Vector Regressor) au lieu d'un `SVC`.